In [91]:
import P04_Parser
from igor2 import binarywave
import numpy as np
import matplotlib.pyplot as plt
import re
import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
import os

def load_ibw_swept_as_kmap(file):
    """Swept (2D) .ibw cut -> Kmap with 3 identical theta_y slices, so P04_Parser plotting works."""
    wave = binarywave.load(file)["wave"]
    data = np.array(wave["wData"], dtype=float)     # (energy, angle)
    sfa = wave["wave_header"]["sfA"]
    sfb = wave["wave_header"]["sfB"]
    energy = sfb[0] + sfa[0] * np.arange(data.shape[0])
    theta = sfb[1] + sfa[1] * np.arange(data.shape[1])
    data_map = np.repeat(data[:, :, np.newaxis], 3, axis=2)
    return P04_Parser.Kmap(data_map, energy, theta, np.array([-1, 0, 1]))

In [92]:
import os
import re

file_dir = r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2B_49eV_LV_temp_series"
files = os.listdir(file_dir)

files_names_fesi = [f for f in files if f.endswith(".ibw") and "testing" not in f and not f.startswith("TaFoil")]

# Sort by temperature, with each "down" scan right after its "up" scan
def sort_key(f):
    T = float(re.search(r"_(\d+(?:\.\d+)?)K", f).group(1))
    return (T, "down" in f.lower())

files_names_fesi = sorted(files_names_fesi, key=sort_key)
files = [os.path.join(file_dir, f) for f in files_names_fesi]

def temp_from_filename(f):
    return str(re.search(r"_(\d+(?:\.\d+)?)K", f).group(1)) + "K" + (" down" if "down" in f.lower() else "")

labels = {f: temp_from_filename(f) for f in files_names_fesi} #[20:-14] for FeSi and [19:-17] for TaFoil
labels

{'FeSi_2B_hv_49_LV_GX_20K_0001Swept.ibw': '20K',
 'FeSi_2B_hv_49_LV_GX_20K_down_0001Swept.ibw': '20K down',
 'FeSi_2B_hv_49_LV_GX_60K_0001Swept.ibw': '60K',
 'FeSi_2B_hv_49_LV_GX_60K_down_0001Swept.ibw': '60K down',
 'FeSi_2B_hv_49_LV_GX_100K_0001Swept.ibw': '100K',
 'FeSi_2B_hv_49_LV_GX_100K_down_0001Swept.ibw': '100K down',
 'FeSi_2B_hv_49_LV_GX_200K_0001Swept.ibw': '200K'}

In [93]:
fermi_levels = np.load("fermi_levels.npy", allow_pickle=True)  # Load the dictionary from the .npy file
fermi_levels = [[x[0].astype(float), x[1].astype(float), x[2]] for x in fermi_levels]  # Convert the first element of each sublist to float

In [94]:
cmap = mpl.colormaps["plasma"]
norm = mpl.colors.Normalize(vmin=0, vmax=7)
temp_to_index = {20: 1, 40: 2, 60: 3, 100: 4, 200: 5}
linestyles = ["-",  "(0, (3, 1, 1, 1))", "--", "-.", ":"]

def line_style(file):
    name = Path(file).name
    T = temp_to_index[int(re.search(r"_(\d+(?:\.\d+)?)K", name).group(1))]
    return dict(
        color=cmap(norm(T)),
        linestyle=linestyles[T - 1] if "down" not in name.lower() else "(0, (3, 1, 1, 1))",
    )


In [105]:
# FeSi EDCs on an E - E_F axis, with E_F taken from the Ta foil fit at the same temperature

keep = ["20K", "60K", "100K", "200K"]
norm_range = (-0.2, -0.1)  # E - E_F (eV); each EDC is divided by its mean intensity in this window

fesi_names = sorted(
    [f for f in os.listdir(file_dir) if f.endswith(".ibw") and "testing" not in f and f.startswith("FeSi")],
    key=sort_key,
)
print(fesi_names)
fesi_names = [f for f in fesi_names if temp_from_filename(f) in keep]
fesi_files = [os.path.join(file_dir, f) for f in fesi_names]
print(fesi_names)
E_F_of = {lab: ef for ef, _, lab in fermi_levels}
# No Ta foil scan exists for 60K (up), so fall back to the mean of the fitted values
E_F_fallback = 44.46525951544912 #- 3e-3#60K measured by hand

fig, ax = plt.subplots(figsize=(14, 8))

for file in fesi_files:
    lab = temp_from_filename(Path(file).name)
    E_F = E_F_of.get(lab, E_F_fallback)
    if lab not in E_F_of:
        print(f"{lab}: no Ta foil fit, using mean E_F = {E_F:.5f} eV")

    data = load_ibw_swept_as_kmap(file)
    edc = data.get_EDC_inside(hs.roi.CircleROI(cx=0, cy=0, r=2))

    energy = edc.axes_manager[0].axis - E_F
    in_range = (energy >= norm_range[0]) & (energy <= norm_range[1])
    intensity = edc.data / np.nanmean(edc.data[in_range])

    ax.plot(
        energy,
        intensity,
        label=lab + ("" if lab in E_F_of else " (mean E_F)"),
        **line_style(file),
    )

ax.axvspan(*norm_range, color="0.85", zorder=0, label="normalization range")
ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(-0.25, 0.25)
ax.set_xlabel(r"$E - E_F$ (eV)")
ax.set_ylabel("Normalized intensity")
ax.set_title("FeSi EDCs, Fermi level from Ta foil")
ax.legend()
plt.show()

['FeSi_2B_hv_49_LV_GX_20K_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_20K_down_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_60K_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_60K_down_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_100K_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_100K_down_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_200K_0001Swept.ibw']
['FeSi_2B_hv_49_LV_GX_20K_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_60K_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_100K_0001Swept.ibw', 'FeSi_2B_hv_49_LV_GX_200K_0001Swept.ibw']
60K: no Ta foil fit, using mean E_F = 44.46526 eV


In [103]:
# %matplotlib qt
from matplotlib.widgets import Slider, Button, SpanSelector

max_shift = 0.02  # eV, slider range is +-max_shift

# Uses keep, norm_range, E_F_of and E_F_fallback from the cell above
edcs = []
for file in files:
    lab = labels[Path(file).name]
    if lab not in keep:
        continue
    E_F = E_F_of.get(lab, E_F_fallback)

    data = load_ibw_swept_as_kmap(file)
    edc = data.get_EDC_inside(hs.roi.CircleROI(cx=0, cy=0, r=2))
    edcs.append((file, edc.axes_manager[0].axis - E_F, edc.data))

# Two files share the "20K" label, so number duplicates to keep slider names unique
slider_labels = []
for file, _, _ in edcs:
    lab = labels[Path(file).name]
    if lab not in E_F_of:
        lab += " (mean E_F)"
    n = sum(l == lab or l.startswith(lab + " #") for l in slider_labels)
    slider_labels.append(lab if n == 0 else f"{lab} #{n + 1}")

slider_height = 0.03
plot_bottom = 0.1 + slider_height * len(edcs)
fig, ax = plt.subplots(figsize=(14, 10))
fig.subplots_adjust(bottom=plot_bottom)

plt.rcParams.update({
    "font.size": 16,          # base size (also used for slider labels and values)
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})


lines = []
for (file, energy, intensity), lab in zip(edcs, slider_labels):
    (line,) = ax.plot(energy, intensity, label=lab, **line_style(file))
    lines.append(line)

ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(-0.25, 0.25)
ax.set_xlabel(r"$E - E_F$ (eV)")
ax.set_ylabel("Normalized intensity")
ax.set_title("FeSi EDCs, Fermi level from Ta foil (drag the grey band to change the normalization range)")
ax.legend()

sliders = []
for i, lab in enumerate(slider_labels):
    slider_ax = fig.add_axes([0.2, plot_bottom - 0.06 - slider_height * (i + 1), 0.6, slider_height * 0.7])
    slider = Slider(
        slider_ax, lab, -max_shift * 1e3, max_shift * 1e3, valinit=0, valstep=0.1,
        valfmt="%+.1f meV", color=line_style(edcs[i][0])["color"],
    )

    sliders.append(slider)

def update(_=None):
    """Shift each curve by its slider, then normalize it to its mean inside the current band."""
    lo, hi = norm_selector.extents
    for (_, energy, intensity), line, slider in zip(edcs, lines, sliders):
        shifted = energy + slider.val * 1e-3
        in_range = (shifted >= lo) & (shifted <= hi)
        line.set_data(shifted, intensity / np.nanmean(intensity[in_range]) if in_range.any() else intensity * np.nan)
    ax.relim()
    ax.autoscale_view(scalex=False)
    fig.canvas.draw_idle()

norm_selector = SpanSelector(
    ax,
    lambda lo, hi: update(),
    "horizontal",
    onmove_callback=lambda lo, hi: update(),
    interactive=True,
    drag_from_anywhere=True,
    props=dict(facecolor="0.6", alpha=0.3),
)
norm_selector.extents = norm_range

for slider in sliders:
    slider.on_changed(update)

reset_ax = fig.add_axes([0.9, 0.02, 0.08, 0.04])
reset_button = Button(reset_ax, "Reset")

def reset(_):
    for s in sliders:
        s.reset()
    norm_selector.extents = norm_range
    update()

reset_button.on_clicked(reset)

def get_shifts():
    """Current slider shift (meV) per curve, keyed by slider label."""
    return {lab: s.val for lab, s in zip(slider_labels, sliders)}

def get_norm_range():
    """Current normalization range (E - E_F, eV)."""
    return norm_selector.extents

update()
plt.show()

In [97]:
E_F_fallback + 1.5 * 1e-3

44.46675951544912

In [100]:
E_F_fallback + 1.5 * 1e-3 - fermi_levels[2][0]

np.float64(0.002016106013961405)